# MIRROR: combine the reproduction runs

Collects the result files written by notebooks 01 to 09 and prints the results
of all four cohorts next to the values reported in the paper. It needs no GPU
and no patient data: attach the outputs of the nine notebooks, and the code.

Code: [https://github.com/Zied-Zaafrani/mirror-drug-rec](https://github.com/Zied-Zaafrani/mirror-drug-rec).

In [ ]:
import json, shutil, statistics, subprocess, sys, zipfile
from pathlib import Path

REPOSITORY_URL = "https://github.com/Zied-Zaafrani/mirror-drug-rec"
INPUT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("./reproduce-output")
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./combined")
REPOSITORY = WORKING / "mirror"
COMBINED = WORKING / "results"
REPORTED = {"mimic3": 0.5706, "mimic4_icu": 0.4664, "mimic4_hospital": 0.5461, "mimic4_mixed": 0.5097}

if not REPOSITORY.exists():
    attached = [p.parent.parent for p in INPUT.rglob("config.yaml")
                if p.parent.name == "src" and (p.parent / "train.py").exists()]
    if attached:
        shutil.copytree(attached[0], REPOSITORY, ignore=shutil.ignore_patterns("results", ".git"))
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "pyyaml"],
               check=True)

# Only the archives written by notebooks 01 to 09 count. The repository's own
# results/ folder holds the reported numbers, so reading it would compare the
# paper with itself.
COMBINED.mkdir(parents=True, exist_ok=True)
archives = sorted(INPUT.rglob("results_*.zip"))
assert archives, "Attach the outputs of notebooks 01 to 09."
for archive in archives:
    print("reading", archive.name)
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(COMBINED)

subprocess.run([sys.executable, str(REPOSITORY / "src" / "report.py"), str(COMBINED)], check=True)

print(f"\n{'Cohort':<18}{'seeds':>6}{'Jaccard here':>22}{'paper':>9}{'difference':>12}")
for cohort, reported in REPORTED.items():
    values = [json.loads(p.read_text())["metrics"]["jaccard"]
              for p in COMBINED.glob(f"{cohort}/full/result_*.json")]
    if not values:
        print(f"{cohort:<18}{0:>6}  not run yet")
        continue
    mean = statistics.fmean(values)
    spread = statistics.stdev(values) if len(values) > 1 else 0.0
    print(f"{cohort:<18}{len(values):>6}{mean:>13.4f} +/- {spread:.4f}{reported:>9.4f}{mean - reported:>+12.4f}")